<a href="https://colab.research.google.com/github/Hadi6618/PRISM/blob/main/ShanghaiTech_Ensemble_Fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# STG-NF + MULDE Frame-Level Ensemble Fusion

This notebook fuses frame-level anomaly scores from **STG-NF** (object/pose level)
and **MULDE** (frame level) for either the **ShanghaiTech Campus** or **Avenue**
dataset.

**Pipeline:**
1. Load the two score pickles produced by the upstream training/export notebooks.
2. Align the two streams per `(video_id, frame_index)`. Video-ID conventions are
   auto-detected and remapped (Avenue: STG-NF `01_0001` <-> MULDE `01`).
3. Apply **global rank** normalization so both streams live on `[0, 1]`.
4. Grid-search the best Gaussian smoothing `sigma` (maximises average standalone AUC).
5. Grid-search the fusion weight `beta_1` over `[0, 1]` in 0.01 steps.
6. Save the grid table (`fusion_grid_search.csv`) and report (`fusion_report.json`)
   to Google Drive.

**Set `DATASET` in the config cell to choose the dataset.** All results are saved
under `/content/drive/MyDrive/Fusion/runs/<DATASET>/ensemble/`.

In [1]:
import subprocess
import importlib.util
import sys
import warnings
from pathlib import Path
import numpy as np
from sklearn.metrics import roc_auc_score

from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/Hadi6618/PRISM.git'
REPO_DIR = Path('/content/PRISM')


Mounted at /content/drive


In [2]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'{REPO_DIR} already exists, skipping clone.')


In [3]:
_FUSION_PATH = REPO_DIR / 'fusion.py'
if not _FUSION_PATH.exists():
    raise FileNotFoundError(f'fusion.py not found at {_FUSION_PATH}')
spec = importlib.util.spec_from_file_location('fusion', _FUSION_PATH)
fusion = importlib.util.module_from_spec(spec)
sys.modules['fusion'] = fusion
spec.loader.exec_module(fusion)
print(f'Loaded fusion module from {_FUSION_PATH}')


Loaded fusion module from /content/PRISM/fusion.py


## Dataset selection

Set `DATASET` to `'ShanghaiTech'` or `'Avenue'`. The paths for each dataset are
pre-configured below — edit them only if your Drive layout differs.

In [12]:
# ======================================================================
# DATASET SELECTOR -- change this one line to switch datasets
# ======================================================================
DATASET = 'Avenue'   # 'ShanghaiTech' or 'Avenue'

# Per-dataset paths on Google Drive. Edit if your layout differs.
DATASET_PATHS = {
    'ShanghaiTech': {
        'stgnf_pkl': Path('/content/drive/MyDrive/STG-NF/original_shanghaitech/logs/shanghaitech_stgnf_scores_84.pkl'),
        'mulde_pkl':  Path('/content/drive/MyDrive/MULDE/runs/shanghaitech_hiera_l_mulde/2026_06_10_04_51_41/artifacts/shanghaitech_mulde_scores_79_7.pkl'),
        'output_dir': Path('/content/drive/MyDrive/Fusion/runs/ShanghaiTech/ensemble'),
    },
    'Avenue': {
        'stgnf_pkl': Path('/content/drive/MyDrive/STG-NF/Avenue_dataset/logs/avenue_stgnf_scores_57.pkl'),
        'mulde_pkl':  Path('/content/drive/MyDrive/MULDE/runs/avenue_hiera_l_mulde/Final_avenue_scores/artifacts/avenue_mulde_scores_81_4.pkl'),
        'output_dir': Path('/content/drive/MyDrive/Fusion/runs/Avenue/ensemble'),
    },
}

assert DATASET in DATASET_PATHS, f'DATASET must be one of {list(DATASET_PATHS.keys())}'
cfg = DATASET_PATHS[DATASET]
STGNF_PKL = cfg['stgnf_pkl']
MULDE_PKL = cfg['mulde_pkl']
OUTPUT_DIR = cfg['output_dir']

for p in [STGNF_PKL, MULDE_PKL]:
    if not p.exists():
        raise FileNotFoundError(f'Missing score file: {p}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Dataset:      {DATASET}')
print(f'STG-NF scores: {STGNF_PKL}')
print(f'MULDE scores:  {MULDE_PKL}')
print(f'Output dir:    {OUTPUT_DIR}')


Dataset:      Avenue
STG-NF scores: /content/drive/MyDrive/STG-NF/Avenue_dataset/logs/avenue_stgnf_scores_57.pkl
MULDE scores:  /content/drive/MyDrive/MULDE/runs/avenue_hiera_l_mulde/Final_avenue_scores/artifacts/avenue_mulde_scores_81_4.pkl
Output dir:    /content/drive/MyDrive/Fusion/runs/Avenue/ensemble


In [13]:
stgnf_scores, stgnf_meta = fusion.load_score_pickle(STGNF_PKL)
mulde_scores, mulde_meta = fusion.load_score_pickle(MULDE_PKL)
print(f'STG-NF videos: {len(stgnf_scores)}  (sample IDs: {list(stgnf_scores.keys())[:3]})')
print(f'MULDE  videos: {len(mulde_scores)}  (sample IDs: {list(mulde_scores.keys())[:3]})')


STG-NF videos: 21  (sample IDs: ['01_0001', '01_0002', '01_0003'])
MULDE  videos: 21  (sample IDs: ['01', '02', '03'])


In [14]:
# Align per (video_id, frame_index). auto_detect_offset sweeps offset in {-2..+2}
# and tries both anomaly/normality polarity for STG-NF, picking the combination
# that maximises STG-NF's standalone Micro AUC. Video-ID aliases are applied
# automatically (Avenue: 01_0001 <-> 01).
with warnings.catch_warnings():
    warnings.simplefilter('ignore', category=RuntimeWarning)
    aligned, align_stats = fusion.align_per_video(
        stgnf_scores,
        mulde_scores,
        auto_detect_offset=True,
        stgnf_score_mode='auto',
    )

n_remap = align_stats.get('video_id_mapping_applied', 0)
print(
    f"Aligned videos: {align_stats['videos_aligned']} "
    f"(STG-NF={align_stats['videos_in_stgnf']}, MULDE={align_stats['videos_in_mulde']})"
)
print(f"  stgnf_frame_offset = {align_stats.get('stgnf_frame_offset', 0)}")
print(f"  stgnf_score_mode   = {align_stats.get('stgnf_score_mode', 'auto')}")
print(f"  video_ids_remapped = {n_remap}")
print(f"  label_inversion    = {align_stats.get('label_inversion_detected', False)}")
if 'auto_detect' in align_stats:
    print('  Offset/polarity candidates:')
    for key, payload in align_stats['auto_detect']['stgnf_micro_auc_per_candidate'].items():
        off, mode = key.split('|', 1)
        auc = payload.get('micro_auc_stgnf')
        auc_s = f'{auc * 100:.4f}%' if auc is not None else 'n/a'
        marker = ' <-- chosen' if (
            int(off) == align_stats.get('stgnf_frame_offset', 0)
            and mode == align_stats.get('stgnf_score_mode', 'auto')
        ) else ''
        print(f'    offset={int(off):+d}  mode={mode:9s}  AUC={auc_s}{marker}')


Aligned videos: 21 (STG-NF=21, MULDE=21)
  stgnf_frame_offset = 0
  stgnf_score_mode   = normality
  video_ids_remapped = 21
  label_inversion    = True
  Offset/polarity candidates:
    offset=-2  mode=anomaly    AUC=43.2796%
    offset=-2  mode=normality  AUC=56.7204%
    offset=-1  mode=anomaly    AUC=43.1558%
    offset=-1  mode=normality  AUC=56.8442%
    offset=+0  mode=anomaly    AUC=43.0448%
    offset=+0  mode=normality  AUC=56.9552% <-- chosen
    offset=+1  mode=anomaly    AUC=42.9164%
    offset=+1  mode=normality  AUC=57.0836%
    offset=+2  mode=anomaly    AUC=42.8017%
    offset=+2  mode=normality  AUC=57.1983%


In [15]:
# Global rank normalization: converts each model's raw scores to [0,1] percentiles.
# Must happen BEFORE Gaussian smoothing so outlier spikes are capped at 1.0.
NORMALIZATION = 'global_rank'
aligned = fusion.apply_normalization(aligned, strategy=NORMALIZATION)
print(f'Normalization strategy: {NORMALIZATION}')

_y = np.concatenate([v.labels for v in aligned])
_s = np.concatenate([v.stgnf_scores for v in aligned])
_m = np.concatenate([v.mulde_scores for v in aligned])
if len(np.unique(_y)) >= 2:
    print(f'STG-NF alone (after norm) Micro AUC: {roc_auc_score(_y, _s) * 100:.4f}%')
    print(f'MULDE  alone (after norm) Micro AUC: {roc_auc_score(_y, _m) * 100:.4f}%')


Normalization strategy: global_rank
STG-NF alone (after norm) Micro AUC: 56.9552%
MULDE  alone (after norm) Micro AUC: 81.3534%


In [16]:
print('Searching for best Gaussian smoothing sigma ...')
best_sigma, sigma_results = fusion.search_best_sigma(
    aligned,
    sigma_candidates=(0, 1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 14, 15),
    normalization=None,  # already normalized above
)
print(f'Best sigma = {best_sigma}')
aligned = fusion.smooth_scores(aligned, sigma=best_sigma)


Searching for best Gaussian smoothing sigma ...
  sigma=  0.0  STG-NF AUC=56.9552%  MULDE AUC=81.3534%  avg=69.1543%
  sigma=  1.0  STG-NF AUC=56.9533%  MULDE AUC=81.5989%  avg=69.2761%
  sigma=  2.0  STG-NF AUC=56.9482%  MULDE AUC=81.7396%  avg=69.3439%
  sigma=  3.0  STG-NF AUC=56.9391%  MULDE AUC=81.8238%  avg=69.3815%
  sigma=  4.0  STG-NF AUC=56.9240%  MULDE AUC=81.8996%  avg=69.4118%
  sigma=  5.0  STG-NF AUC=56.9015%  MULDE AUC=81.9764%  avg=69.4389%
  sigma=  6.0  STG-NF AUC=56.8738%  MULDE AUC=82.0531%  avg=69.4635%
  sigma=  8.0  STG-NF AUC=56.7901%  MULDE AUC=82.1992%  avg=69.4947%
  sigma= 10.0  STG-NF AUC=56.6829%  MULDE AUC=82.3240%  avg=69.5034%
  sigma= 11.0  STG-NF AUC=56.6336%  MULDE AUC=82.3807%  avg=69.5071%
  sigma= 12.0  STG-NF AUC=56.5784%  MULDE AUC=82.4300%  avg=69.5042%
  sigma= 13.0  STG-NF AUC=56.5243%  MULDE AUC=82.4712%  avg=69.4977%
  sigma= 14.0  STG-NF AUC=56.4659%  MULDE AUC=82.5088%  avg=69.4874%
  sigma= 15.0  STG-NF AUC=56.4052%  MULDE AUC=82.5371% 

In [17]:
beta_1_values = list(np.round(np.arange(0.0, 1.0 + 1e-9, 0.01), 4))
results, best, summary = fusion.grid_search_fusion(aligned, beta_1_values=beta_1_values)
summary['dataset'] = DATASET
summary['normalization'] = NORMALIZATION
summary['smooth_sigma'] = best_sigma
fusion.write_outputs(
    results=results,
    best=best,
    summary=summary,
    alignment_stats=align_stats,
    stgnf_meta=stgnf_meta,
    mulde_meta=mulde_meta,
    output_dir=OUTPUT_DIR,
)


Saved grid search table: /content/drive/MyDrive/Fusion/runs/Avenue/ensemble/fusion_grid_search.csv
Saved ensemble report:   /content/drive/MyDrive/Fusion/runs/Avenue/ensemble/fusion_report.json
Optimal weights -> beta_1 (STG-NF)=0.08, beta_2 (MULDE)=0.92, Micro AUC=82.5247%


In [18]:
import pandas as pd
table = fusion.results_to_table(results)
table_sorted = table.dropna(subset=['micro_auc']).sort_values('micro_auc', ascending=False)
display(table_sorted.head(30))


,beta_1_stgnf,beta_2_mulde,micro_auc
8,0.08,0.92,0.825247
9,0.09,0.91,0.825213
7,0.07,0.93,0.825212
6,0.06,0.94,0.825132
10,0.10,0.90,0.825077
5,0.05,0.95,0.825007
11,0.11,0.89,0.824953
4,0.04,0.96,0.824817
12,0.12,0.88,0.824792
13,0.13,0.87,0.824635


In [19]:
y = np.concatenate([v.labels for v in aligned])
s = np.concatenate([v.stgnf_scores for v in aligned])
m = np.concatenate([v.mulde_scores for v in aligned])
print(f'Dataset:    {DATASET}')
print(f'Frames:     {y.shape[0]}')
print(f'Videos:     {len(aligned)}')
print(f'STG-NF AUC: {roc_auc_score(y, s) * 100:.4f}%')
print(f'MULDE  AUC: {roc_auc_score(y, m) * 100:.4f}%')
print(f'Sigma:      {best_sigma}')
if best is not None and best.micro_auc is not None:
    print(f'Ensemble:   {best.micro_auc * 100:.4f}% (beta_1={best.beta_1:.2f}, beta_2={best.beta_2:.2f})')
print(f'Correlation (STG-NF vs MULDE): {np.corrcoef(s, m)[0, 1]:.4f}')


Dataset:    Avenue
Frames:     15324
Videos:     21
STG-NF AUC: 56.6336%
MULDE  AUC: 82.3807%
Sigma:      11.0
Ensemble:   82.5247% (beta_1=0.08, beta_2=0.92)
Correlation (STG-NF vs MULDE): 0.0990
